In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.2 Memory accounting

Serving memory is weights + activations + **KV cache**. The cache is the part that
grows with batch size and sequence length, and it is what runs you out of memory
first. Learn to estimate it before you deploy.

In [ ]:
from pocketlm import PocketLM, PocketLMConfig

cfg = PocketLMConfig(vocab_size=100, block_size=512, n_layer=6, n_head=8, n_embd=512)
params = sum(p.numel() for p in PocketLM(cfg).parameters())
param_mb = params * 4 / 1e6                      # fp32 weights
head_size = cfg.n_embd // cfg.n_head
seq, batch = 512, 1
kv_mb = 2 * cfg.n_layer * cfg.n_head * seq * head_size * batch * 4 / 1e6
print(f"weights: {params / 1e6:.1f}M params = {param_mb:.1f} MB (fp32)")
print(f"KV cache @ seq={seq}, batch={batch}: {kv_mb:.2f} MB")

Double the batch or the sequence and the cache doubles. This is exactly why GQA
(fewer KV heads) and quantization (fewer bits) matter, they attack this term.

In [ ]:
assert abs(param_mb - params * 4 / 1e6) < 1e-9
assert kv_mb > 0